# Roadmap 1 — Basic Chatbot

Build a basic conversational chatbot using:

- Python
- OpenAI API
- Gradio
- Gradio State

The chatbot will eventually support:

- Chat interface
- OpenAI model responses
- Conversation history
- Per-session state
- Token usage tracking
- Cost estimation
- Error handling
- Clear conversation

## 1. Create the Gradio Chat Interface

First, create the chatbot interface without connecting it to OpenAI.

The goal is to verify that:

- The chat window works
- The user can enter a message
- The interface can return a response
- The interface can be cleared
- Token usage and cost sections are visible

For now, the chatbot will return a fixed response.

In [ ]:
import gradio as gr
from dotenv import load_dotenv
import os

from openai import OpenAI

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = "gpt-4.1-mini"

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
print("API key loaded:", bool(OPENAI_API_KEY))
print("Model:", MODEL_NAME)

In [ ]:
def chatbot(message, history):
    return "Hello! This is a test response."

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot",
    description="My first chatbot :p"
)

demo.launch()

In [ ]:
demo.close()

## 2. Make the First OpenAI Request

Now that the Gradio interface works, connect the chatbot to OpenAI.

The first request will contain:

1. A system message describing the assistant
2. The user's current message

The model's response will be returned to the interface.

In [ ]:
def chatbot(message, history):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": message
        }
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot"
)

demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

## 3. Add Conversation History

The previous implementation only sends the current user message.

To allow the model to remember previous messages, create a Python list containing the complete conversation.

The list will contain:

- System message
- User messages
- Assistant messages

Each new message is appended in the order it occurs.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    }
]

In [ ]:
def chatbot(message, history):

    messages.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot",
    description="My first chatbot :p"
)

demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

In [ ]:
messages

## 4. Add Gradio State

A global `messages` list would be shared by everyone using the application.

Instead, use `gr.State` to maintain a separate conversation for each Gradio session.

Each session will have its own:

- Conversation history
- Token usage
- Cost information

In [ ]:
def create_initial_state():

    return {
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful assistant."
            }
        ],
        "usage": {
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "total_cost": 0
        }
    }

In [ ]:
state = gr.State(create_initial_state())

In [ ]:
def chatbot(message, history, state):

    messages = state["messages"]

    messages.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reply
    })

    return reply, state

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot",
    description="My first chatbot :p",
    additional_inputs=[state],
    additional_outputs=[state]
)

In [ ]:
demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

## 5. Track Token Usage

The OpenAI response contains token usage information.

Track:

- Input tokens
- Output tokens
- Total tokens

Two levels of usage will be displayed:

### Current Request
Tokens used by the latest API request.

### Session
Total tokens accumulated across the current conversation.

In [ ]:
test_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Hello!"
        }
    ]
)

In [ ]:
input_tokens = test_response.usage.prompt_tokens
output_tokens = test_response.usage.completion_tokens
total_tokens = test_response.usage.total_tokens

print("Input:", input_tokens)
print("Output:", output_tokens)
print("Total:", total_tokens)

In [ ]:
def chatbot(message, history, state):

    messages = state["messages"]
    usage = state["usage"]

    messages.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reply
    })

    # Current request
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens

    # Session totals
    usage["input_tokens"] += input_tokens
    usage["output_tokens"] += output_tokens
    usage["total_tokens"] += total_tokens

    print("Current Request")
    print("Input:", input_tokens)
    print("Output:", output_tokens)
    print("Total:", total_tokens)

    print("\nSession")
    print("Input:", usage["input_tokens"])
    print("Output:", usage["output_tokens"])
    print("Total:", usage["total_tokens"])

    return reply, state

In [ ]:
state = gr.State(create_initial_state())

demo = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot",
    description="My first chatbot :p",
    additional_inputs=[state],
    additional_outputs=[state]
)

demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

## 6. Add Model Pricing

Create a pricing configuration for the selected model.

The pricing configuration stores:

- Input price per 1 million tokens
- Output price per 1 million tokens

The chatbot will calculate:

- Input cost
- Output cost
- Total request cost
- Total session cost

In [ ]:
MODEL_PRICING = {
    "gpt-4.1-mini": {
        "input": 0.40,
        "output": 1.60,
    },
    "gpt-4.1": {
        "input": 2.00,
        "output": 8.00,
    },
    "gpt-4o-mini": {
        "input": 0.15,
        "output": 0.60,
    },
}

In [ ]:
def calculate_cost(model, input_tokens, output_tokens):

    pricing = MODEL_PRICING[model]

    input_cost = (
        input_tokens / 1_000_000
    ) * pricing["input"]

    output_cost = (
        output_tokens / 1_000_000
    ) * pricing["output"]

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

In [ ]:
cost = calculate_cost(
    MODEL_NAME,
    1000,
    500
)

cost

In [ ]:
def chatbot(message, history, state):

    messages = state["messages"]
    usage = state["usage"]

    messages.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reply
    })

    # Current request tokens
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens

    # Current request cost
    cost = calculate_cost(
        MODEL_NAME,
        input_tokens,
        output_tokens
    )

    # Session totals
    usage["input_tokens"] += input_tokens
    usage["output_tokens"] += output_tokens
    usage["total_tokens"] += total_tokens
    usage["total_cost"] += cost["total_cost"]

    print("Current Request")
    print("Input:", input_tokens)
    print("Output:", output_tokens)
    print("Total:", total_tokens)

    print("Input Cost:", cost["input_cost"])
    print("Output Cost:", cost["output_cost"])
    print("Total Cost:", cost["total_cost"])

    print("\nSession")
    print("Total Tokens:", usage["total_tokens"])
    print("Total Cost:", usage["total_cost"])

    return reply, state

In [ ]:
def get_request_stats(
    input_tokens=0,
    output_tokens=0,
    total_tokens=0,
    cost=None
):

    if cost is None:
        cost = {
            "input_cost": 0,
            "output_cost": 0,
            "total_cost": 0,
        }

    return f"""
### Current Request

**Tokens**

- Input: **{input_tokens}**
- Output: **{output_tokens}**
- Total: **{total_tokens}**

**Cost**

- Input: **${cost['input_cost']:.6f}**
- Output: **${cost['output_cost']:.6f}**
- Total: **${cost['total_cost']:.6f}**
"""

In [ ]:
def get_session_stats(usage):

    return f"""
### Session

**Tokens**

- Input: **{usage['input_tokens']}**
- Output: **{usage['output_tokens']}**
- Total: **{usage['total_tokens']}**

**Cost**

- Total: **${usage['total_cost']:.6f}**
"""

In [ ]:
def chatbot(message, history, state):

    messages = state["messages"]
    usage = state["usage"]

    messages.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reply
    })

    # Token usage
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens

    # Cost
    cost = calculate_cost(
        MODEL_NAME,
        input_tokens,
        output_tokens
    )

    # Session totals
    usage["input_tokens"] += input_tokens
    usage["output_tokens"] += output_tokens
    usage["total_tokens"] += total_tokens
    usage["total_cost"] += cost["total_cost"]

    request_stats = get_request_stats(
        input_tokens,
        output_tokens,
        total_tokens,
        cost
    )

    session_stats = get_session_stats(usage)

    return (
        reply,
        state,
        request_stats,
        session_stats
    )

In [ ]:
state = gr.State(create_initial_state())

request_stats = gr.Markdown(
    get_request_stats()
)

session_stats = gr.Markdown(
    get_session_stats(create_initial_state()["usage"])
)

demo = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot",
    description="My first chatbot :p",
    additional_inputs=[state],
    additional_outputs=[
        state,
        request_stats,
        session_stats
    ]
)

In [ ]:
demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

## 7. Add Error Handling

The chatbot should handle common failures without crashing.

Handle:

- Missing API key
- Invalid API key
- Network failure
- Rate limit
- Model timeout
- Invalid request
- Empty user message

The interface should display a readable error message instead of a Python traceback.

In [ ]:
from openai import (
    AuthenticationError,
    RateLimitError,
    APITimeoutError,
    APIConnectionError,
    BadRequestError,
)

In [ ]:
def chatbot(message, history, state):

    messages = state["messages"]
    usage = state["usage"]

    # Empty message
    if not message or not message.strip():
        return (
            "⚠️ Please enter a message before sending.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    # Missing API key
    if not OPENAI_API_KEY:
        return (
            "❌ OpenAI API key is missing.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    messages.append({
        "role": "user",
        "content": message
    })

    try:

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages
        )

        reply = response.choices[0].message.content

        messages.append({
            "role": "assistant",
            "content": reply
        })

        # Token usage
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        total_tokens = response.usage.total_tokens

        # Cost
        cost = calculate_cost(
            MODEL_NAME,
            input_tokens,
            output_tokens
        )

        # Session totals
        usage["input_tokens"] += input_tokens
        usage["output_tokens"] += output_tokens
        usage["total_tokens"] += total_tokens
        usage["total_cost"] += cost["total_cost"]

        request_stats = get_request_stats(
            input_tokens,
            output_tokens,
            total_tokens,
            cost
        )

        session_stats = get_session_stats(usage)

        return (
            reply,
            state,
            request_stats,
            session_stats
        )

    except AuthenticationError:
        return (
            "❌ Invalid OpenAI API key.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    except RateLimitError:
        return (
            "⚠️ Rate limit exceeded. Please try again later.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    except APIConnectionError:
        return (
            "🌐 Unable to connect to OpenAI. Please check your internet connection.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    except APITimeoutError:
        return (
            "⌛ The request timed out. Please try again.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    except BadRequestError:
        return (
            "❌ Invalid request sent to OpenAI.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

    except Exception as e:
        print("Unexpected error:", e)

        return (
            "❌ Something went wrong. Please try again.",
            state,
            get_request_stats(),
            get_session_stats(usage)
        )

## 8. Clear Conversation

The clear action should:

- Remove all user messages
- Remove all assistant messages
- Retain the original system message
- Reset input token totals
- Reset output token totals
- Reset total token count
- Reset estimated cost

In [ ]:
def clear_conversation():

    new_state = create_initial_state()

    return (
        [],
        new_state,
        get_request_stats(),
        get_session_stats(new_state["usage"])
    )

In [ ]:
state = gr.State(create_initial_state())

In [ ]:
request_stats = gr.Markdown(
    get_request_stats()
)

session_stats = gr.Markdown(
    get_session_stats(
        create_initial_state()["usage"]
    )
)

In [ ]:
chat_interface = gr.ChatInterface(
    fn=chatbot,
    title="Framework Free Chatbot",
    description="My first chatbot :p",
    additional_inputs=[state],
    additional_outputs=[
        state,
        request_stats,
        session_stats
    ]
)

In [ ]:
clear_btn = gr.Button("🗑️ Clear Conversation")

In [ ]:
with gr.Blocks() as demo:

    state = gr.State(create_initial_state())

    request_stats = gr.Markdown(
        get_request_stats()
    )

    session_stats = gr.Markdown(
        get_session_stats(
            create_initial_state()["usage"]
        )
    )

    chat_interface = gr.ChatInterface(
        fn=chatbot,
        title="Framework Free Chatbot",
        description="My first chatbot :p",
        additional_inputs=[state],
        additional_outputs=[
            state,
            request_stats,
            session_stats
        ]
    )

    clear_btn = gr.Button(
        "🗑️ Clear Conversation"
    )

    clear_btn.click(
        clear_conversation,
        outputs=[
            chat_interface.chatbot,
            state,
            request_stats,
            session_stats
        ]
    )

In [ ]:
demo.launch()

In [ ]:
demo.close()